# ClosureWatch — Restaurant Closure Prediction

**Problem:** Given 12 months of Yelp behavioral signals for a restaurant, predict whether it will permanently close in the next 6 months.

**Framing:** SME credit underwriting — alternative lenders use public behavioral signals the same way banks use transaction history. If you can predict business failure from Yelp activity alone, you can flag deteriorating credits before they default.

**Dataset:** Yelp Academic Dataset — 9 US metros (Tampa, Philadelphia, Indianapolis, Tucson, Nashville, New Orleans, Saint Louis, Reno, Boise), 18,464 restaurants with valid feature windows, 9.4% overall closure rate.

**GitHub (full pipeline):** https://github.com/leonardo-schneider/Clousurewatch

---

*This notebook loads the pre-processed feature files directly from the GitHub repo and reproduces the modeling pipeline, evaluation, and figures. The raw data pipeline (loading Yelp JSON, building labels, engineering features) lives in the GitHub scripts.*

## Setup

In [ ]:
!pip install shap -q

import os, sys, warnings, json
warnings.filterwarnings('ignore')

if not os.path.exists('Clousurewatch'):
    !git clone https://github.com/leonardo-schneider/Clousurewatch.git --quiet

os.chdir('Clousurewatch')
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap
from pathlib import Path
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, roc_curve
)
from app_helpers import add_null_flags

plt.rcParams.update({
    'font.family': 'serif', 'figure.dpi': 120,
    'axes.spines.top': False, 'axes.spines.right': False,
})
print('Setup complete.')

## 1. Load Data

Features are pre-computed for each metro and stored as small Parquet files (~0.2–0.6 MB each). The raw Yelp JSON (~10 GB) is not needed here.

**Data reduction funnel — from raw Yelp to model-ready dataset:**

| Stage | Count | Notes |
|---|---|---|
| All businesses in Yelp dataset | 150,346 | All categories, all cities |
| In the 9 target cities | ~53,375 | City/state match per metro |
| Restaurant or food category | ~31,627 | Drops bars-only, grocery stores, etc. |
| Valid anchor date 2016–2020 | **18,464** | Main reduction: businesses must have enough review history to compute a 12-month observation window with a resolved label |

The largest drop (31,627 → 18,464) happens during label-building, not loading. Restaurants are excluded if their 80th-percentile review date falls outside the 2016–2020 window, meaning they opened too recently or have too sparse a history to form a valid observation period.

In [ ]:
LATEST_ANCHOR = pd.Timestamp('2020-06-01')
TEST_FRAC     = 0.20
TARGET_COL    = 'closed_within_6m'
META_COLS     = {'business_id', TARGET_COL, 'anchor_date', 'city', 'state', 'metro'}

METRO_DIRS = {
    'tampa':         Path('data/processed'),
    'philadelphia':  Path('data/processed_philly'),
    'indianapolis':  Path('data/processed_indianapolis'),
    'tucson':        Path('data/processed_tucson'),
    'nashville':     Path('data/processed_nashville'),
    'new_orleans':   Path('data/processed_new_orleans'),
    'saint_louis':   Path('data/processed_saint_louis'),
    'reno':          Path('data/processed_reno'),
    'boise':         Path('data/processed_boise'),
}

frames = []
for metro, d in METRO_DIRS.items():
    df = pd.read_parquet(d / 'features.parquet')
    df = add_null_flags(df)
    emb = d / 'review_embeddings.parquet'
    if emb.exists():
        df = df.merge(pd.read_parquet(emb), on='business_id', how='left')
    df['metro'] = metro
    df['anchor_date'] = pd.to_datetime(df['anchor_date'])
    frames.append(df)

all_df = pd.concat(frames, ignore_index=True)
all_df = all_df[all_df['anchor_date'] <= LATEST_ANCHOR].copy()

n_total  = len(all_df)
n_closed = all_df[TARGET_COL].sum()
print(f'Total restaurants : {n_total:,}')
print(f'Closed within 6m  : {int(n_closed):,} ({n_closed/n_total:.1%})')
print(f'Anchor date range : {all_df["anchor_date"].min().date()} to {all_df["anchor_date"].max().date()}')
print(f'Features          : {sum(1 for c in all_df.columns if c not in META_COLS)}')

## 2. Exploratory Data Analysis

In [ ]:
# Dataset composition by metro
metro_stats = all_df.groupby('metro')[TARGET_COL].agg(['mean', 'sum', 'count']).reset_index()
metro_stats = metro_stats.sort_values('mean')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].barh(metro_stats['metro'], metro_stats['mean'] * 100, color='#2E86AB', alpha=0.85)
axes[0].axvline(all_df[TARGET_COL].mean() * 100, color='gray', ls='--', lw=1.2,
                label=f"Overall {all_df[TARGET_COL].mean():.1%}")
axes[0].set_xlabel('Closure Rate (%)')
axes[0].set_title('Closure Rate by Metro', fontweight='bold')
axes[0].legend(fontsize=9)

axes[1].barh(metro_stats['metro'], metro_stats['count'], color='#E84855', alpha=0.85)
axes[1].set_xlabel('Number of Restaurants')
axes[1].set_title('Dataset Size by Metro', fontweight='bold')

plt.tight_layout()
plt.show()

display(
    metro_stats.rename(columns={'mean': 'closure_rate', 'sum': 'n_closed', 'count': 'n_restaurants'})
    .assign(closure_rate=lambda x: x['closure_rate'].map('{:.1%}'.format),
            n_closed=lambda x: x['n_closed'].astype(int))
    .reset_index(drop=True)
)

### Feature Definitions

Before looking at distributions, here is what each behavioral signal means. All features are computed using only data **strictly before the anchor date** (no leakage).

| Feature | What it measures | Expected signal |
|---|---|---|
| **Days Since Last Review** | Number of days between the most recent review in the observation window and the anchor date. A restaurant that stopped getting reviews long before the anchor date is essentially invisible on Yelp. | Closed restaurants tend to go silent earlier — higher values predict closure |
| **Reviews per Month** | Average number of new Yelp reviews received per month during the 12-month observation window. Measures how actively the restaurant is being reviewed. | Closed restaurants tend to have lower velocity — fewer reviews per month |
| **Mean VADER Sentiment** | Average sentiment score of review text, computed with VADER (a rule-based NLP model tuned for social media). Scores range from -1 (very negative) to +1 (very positive). | Closed restaurants tend to skew slightly more negative, but the separation is weaker than recency signals |
| **Review Momentum** | Ratio of reviews received in the **last 6 months** of the observation window to reviews in the **first 6 months** (+ 1 to avoid division by zero). Value of 1 = stable activity, < 1 = declining, > 1 = growing. | Closed restaurants tend to have momentum < 1 — they were losing engagement before closing |

The core intuition is **"going silent on Yelp is a leading indicator of closure."** A restaurant that stops receiving reviews, or whose review pace is slowing, is showing behavioral warning signs before it actually closes.

In [ ]:
# Distribution of key signals: open vs closed
signals = [
    ('days_since_last_review', 'Days Since Last Review',              (0, 600)),
    ('review_velocity',        'Reviews per Month',                    (0, 15)),
    ('mean_vader',             'Mean VADER Sentiment (-1 to +1)',      (-0.5, 1.0)),
    ('review_momentum',        'Review Momentum (last 6m / first 6m)', (0, 5)),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (col, label, xlim) in zip(axes, signals):
    for label_val, color, name in [(0, '#2E86AB', 'Open'), (1, '#E84855', 'Closed')]:
        data = all_df[all_df[TARGET_COL] == label_val][col].dropna().clip(*xlim)
        med  = data.median()
        ax.hist(data, bins=40, alpha=0.6, color=color, label=f'{name} (med={med:.1f})', density=True)
    ax.set_xlabel(label, fontsize=8)
    ax.set_ylabel('Density')
    ax.set_title(label.split('(')[0].strip(), fontweight='bold', fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle('Key Behavioral Signals: Open vs Closed Restaurants\n'
             '(distributions clipped at 95th percentile for readability)',
             fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()

## 3. Modeling Pipeline

### Temporal Design (Anti-Leakage)

```
|--- obs_start ---|--- OBSERVATION WINDOW (12m) ---|--- anchor_date ---|--- OUTCOME (6m) ---|
                          Features built here               ^                  Label here
                                                 80th percentile review date
```

Every design decision prevents data leakage:

1. **Anchor date = 80th percentile review date** — not the last review. Using the last review would bias the model toward restaurants still actively reviewed at prediction time.
2. **All features use only data strictly before the anchor date.**
3. **Labels come from review recency, not Yelp's `is_open` field** — which reflects 2022 download status, not the prediction date.
4. **Train/test split is time-based** (80/20 by anchor date). A random split would mix restaurants from different time periods, giving an overly optimistic evaluation that does not reflect deployment — where the model always scores restaurants more recent than any it was trained on.
5. **Imputation medians fit on training set only**, then applied to test.

### Models

| Model | Setup | Hyperparameter search |
|---|---|---|
| Logistic Regression | `class_weight='balanced'`, StandardScaler | C ∈ {0.01, 0.1, 1.0, 10.0} |
| XGBoost | `scale_pos_weight=10` | 36 combos: n_estimators, max_depth, learning_rate, min_child_weight |

Best parameters selected by **5-fold StratifiedKFold CV** on training set, primary metric AUC-PR.

### Why AUC-PR?

With a ~9% positive rate, accuracy is meaningless (predicting all-negative gives 91% accuracy). AUC-ROC is insensitive to class imbalance. **AUC-PR directly measures performance on the minority class** and is the correct primary metric here. Random baseline AUC-PR ≈ closure rate ≈ 0.09.

In [ ]:
# Time-based 80/20 split
all_df_s = all_df.sort_values('anchor_date').reset_index(drop=True)
n_test   = max(1, int(len(all_df_s) * TEST_FRAC))
train_df = all_df_s.iloc[:-n_test].copy()
test_df  = all_df_s.iloc[-n_test:].copy()

feat_cols = [c for c in train_df.columns if c not in META_COLS]
medians   = train_df[feat_cols].median()
X_train   = train_df[feat_cols].fillna(medians).values
y_train   = train_df[TARGET_COL].values
X_test    = test_df[feat_cols].fillna(medians).values
y_test    = test_df[TARGET_COL].values

print(f'Training set : {len(train_df):,} restaurants | closure rate {y_train.mean():.1%} | anchors {train_df["anchor_date"].min().date()} -> {train_df["anchor_date"].max().date()}')
print(f'Test set     : {len(test_df):,}  restaurants | closure rate {y_test.mean():.1%}  | anchors {test_df["anchor_date"].min().date()} -> {test_df["anchor_date"].max().date()}')
print(f'Features     : {len(feat_cols)}')
print()
print('Note: the test set has a lower closure rate (6.2%) than training (10.1%).')
print('This is a real temporal shift -- later cohorts have fewer confirmed closures in the Yelp snapshot.')

In [ ]:
# Load models (trained by 05_modeling.py: 5-fold CV selected best params,
# then retrained on the full 80% training set)
xgb    = joblib.load('models/xgboost_final.pkl')
lr     = joblib.load('models/logistic_regression_final.pkl')
scaler = joblib.load('models/lr_scaler_final.pkl')
results = json.load(open('models/final_results.json'))

print('XGBoost best params (selected by 5-fold CV on training set):')
for k, v in results['xgboost']['best_params'].items():
    print(f'  {k}: {v}')
print(f'\nLogistic Regression best C: {results["logistic_regression"]["best_C"]}')

prob_xgb_train = xgb.predict_proba(X_train)[:, 1]
prob_xgb_test  = xgb.predict_proba(X_test)[:, 1]
prob_lr_train  = lr.predict_proba(scaler.transform(X_train))[:, 1]
prob_lr_test   = lr.predict_proba(scaler.transform(X_test))[:, 1]
print('\nProbabilities computed.')

## 4. Results

In [ ]:
r = results
rows = [
    {
        'Model':         'Logistic Regression',
        'CV AUC-PR':     f"{r['logistic_regression']['cv']['AUC_PR']:.3f} +/- {r['logistic_regression']['cv']['std']:.3f}",
        'Train AUC-PR':  f"{r['logistic_regression']['train']['AUC_PR']:.3f}",
        'Test AUC-PR':   f"{r['logistic_regression']['test']['AUC_PR']:.3f}",
        'Train AUC-ROC': f"{r['logistic_regression']['train']['AUC_ROC']:.3f}",
        'Test AUC-ROC':  f"{r['logistic_regression']['test']['AUC_ROC']:.3f}",
        'Test F1':       f"{r['logistic_regression']['test']['F1']:.3f}",
        'Threshold':     f"{r['logistic_regression']['test']['threshold']:.3f}",
    },
    {
        'Model':         'XGBoost',
        'CV AUC-PR':     f"{r['xgboost']['cv']['AUC_PR']:.3f} +/- {r['xgboost']['cv']['std']:.3f}",
        'Train AUC-PR':  f"{r['xgboost']['train']['AUC_PR']:.3f}",
        'Test AUC-PR':   f"{r['xgboost']['test']['AUC_PR']:.3f}",
        'Train AUC-ROC': f"{r['xgboost']['train']['AUC_ROC']:.3f}",
        'Test AUC-ROC':  f"{r['xgboost']['test']['AUC_ROC']:.3f}",
        'Test F1':       f"{r['xgboost']['test']['F1']:.3f}",
        'Threshold':     f"{r['xgboost']['test']['threshold']:.3f}",
    },
]
print('Primary metric: AUC-PR (correct for ~9% imbalanced data; random baseline ~ 0.09)\n')
display(pd.DataFrame(rows))

In [ ]:
# Precision-Recall Curves: both models, training and test side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (y, p_xgb, p_lr, split_label) in zip(axes, [
    (y_train, prob_xgb_train, prob_lr_train, 'Training'),
    (y_test,  prob_xgb_test,  prob_lr_test,  'Test'),
]):
    baseline = y.mean()
    px, rx, _ = precision_recall_curve(y, p_xgb)
    pl, rl, _ = precision_recall_curve(y, p_lr)
    ax.plot(rx, px, color='#2E86AB', lw=2,
            label=f"XGBoost  AP={average_precision_score(y, p_xgb):.3f}")
    ax.plot(rl, pl, color='#E84855', lw=2, ls='--',
            label=f"Logistic Reg  AP={average_precision_score(y, p_lr):.3f}")
    ax.axhline(baseline, color='gray', ls=':', lw=1.2, alpha=0.7,
               label=f'Random baseline ({baseline:.2f})')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title(f'PR Curve -- {split_label} Set\n(n={len(y):,}, closure rate={baseline:.1%})',
                 fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Precision-Recall Curves: XGBoost vs Logistic Regression', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves: both models, training and test side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (y, p_xgb, p_lr, split_label) in zip(axes, [
    (y_train, prob_xgb_train, prob_lr_train, 'Training'),
    (y_test,  prob_xgb_test,  prob_lr_test,  'Test'),
]):
    fpr_x, tpr_x, _ = roc_curve(y, p_xgb)
    fpr_l, tpr_l, _ = roc_curve(y, p_lr)
    ax.plot(fpr_x, tpr_x, color='#2E86AB', lw=2,
            label=f"XGBoost  AUC={roc_auc_score(y, p_xgb):.3f}")
    ax.plot(fpr_l, tpr_l, color='#E84855', lw=2, ls='--',
            label=f"Logistic Reg  AUC={roc_auc_score(y, p_lr):.3f}")
    ax.plot([0, 1], [0, 1], color='gray', ls='--', lw=1, alpha=0.5, label='Random')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title(f'ROC Curve -- {split_label} Set\n(n={len(y):,}, closure rate={y.mean():.1%})',
                 fontweight='bold')
    ax.legend(fontsize=9, loc='lower right')

plt.suptitle('ROC Curves: XGBoost vs Logistic Regression', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Feature Importance (SHAP)

SHAP (SHapley Additive exPlanations) decomposes each prediction into per-feature contributions. Each dot in the beeswarm is one restaurant:
- **Position on X axis** — how much that feature pushed the prediction toward closure (right) or away from it (left)
- **Color** — the feature value for that restaurant (red = high, blue = low)

For example: a red dot on the right for `days_since_last_review` means a restaurant with a **high** recency gap was pushed **toward** a closure prediction — exactly what we expect.

In [ ]:
explainer = shap.TreeExplainer(xgb)
rng = np.random.default_rng(42)
idx = rng.choice(len(X_train), size=min(2000, len(X_train)), replace=False)
shap_values = explainer.shap_values(X_train[idx])

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, X_train[idx],
    feature_names=feat_cols,
    max_display=20,
    show=False,
)
plt.title('SHAP Feature Importance -- XGBoost (sample n=2,000)', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Results Summary

| Model | CV AUC-PR | Test AUC-PR | Test AUC-ROC | Test F1 |
|---|---|---|---|---|
| Logistic Regression (benchmark) | 0.355 +/- 0.054 | 0.245 | 0.808 | 0.336 |
| **XGBoost** | **0.400 +/- 0.061** | **0.328** | **0.833** | **0.382** |
| Random baseline | ~0.094 | — | 0.500 | — |

XGBoost outperforms the logistic regression benchmark across all metrics on the held-out test set, achieving roughly **3.5x lift over random** on AUC-PR.

### Train-Test Gap

XGBoost shows a meaningful AUC-PR drop from training (0.562) to test (0.328). Two factors contribute:

1. **Model variance** — gradient-boosted trees with aggressive imbalance weighting tend to memorize minority-class patterns in training.
2. **Temporal distribution shift** — the test set has a lower closure rate (6.2% vs 10.1% in train) because it contains later anchor dates closer to 2020. AUC-PR is sensitive to base rate changes, so part of the gap reflects this real-world shift rather than pure overfitting.

The AUC-ROC gap is smaller (0.898 → 0.833), suggesting the model's **ranking generalizes well** — it's the calibration that degrades most. For the underwriting use case (rank-order risk, not calibrated probability), AUC-ROC is the more operationally relevant metric.

### Top Predictors (SHAP)

The strongest signals are all variants of **going silent on Yelp**:
- `days_since_last_review` — long recency gap strongly predicts closure
- `review_drought_flag` — 90+ day silence before anchor date
- `review_velocity` — slowing review rate
- `review_momentum` — declining late-window vs early-window activity

Rating and sentiment features contribute but are secondary — a restaurant can maintain its star rating while already showing silence signals before closing.

### Known Limitation

~12% of restaurants have fewer than 18 months of total review history, leaving the 12-month observation window partially empty. These restaurants receive low-confidence predictions rather than incorrect ones — XGBoost handles informative missingness natively through its missing-value splits.